#  01 Known-Parameter Inference — ALARM
The known-parameter experiment follows a fixed procedure: exact ground truth from the original ALARM network $\rightarrow$ exact localization on the four frozen $H_R$ graphs $\rightarrow$ FGCR conditional queries, with the original induced subgraph $G_R$ used as the IC structural-misspecification baseline

## 1. Load the Shared Core from Notebook 00


In [1]:
import json
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
COMMON_CORE_PATH = PROJECT_ROOT / "00_FGCR_Common_Core.ipynb"
if not COMMON_CORE_PATH.exists():
    raise FileNotFoundError("Missing 00_FGCR_Common_Core.ipynb in the project root")

_common_notebook = json.loads(COMMON_CORE_PATH.read_text(encoding="utf-8"))
_loaded_common_cells = 0
for _cell in _common_notebook["cells"]:
    _tags = _cell.get("metadata", {}).get("tags", [])
    if _cell.get("cell_type") == "code" and "common_core" in _tags:
        exec(compile("".join(_cell.get("source", [])), str(COMMON_CORE_PATH), "exec"), globals())
        _loaded_common_cells += 1

if _loaded_common_cells != 7:
    raise RuntimeError(f"Expected 7 common_core cells, loaded {_loaded_common_cells}")


## 2. Known-Parameter Computation Core


In [2]:
# Set the environment variable FGCR_RUN_MODE=formal before the formal run.
RUN_MODE = os.environ.get("FGCR_RUN_MODE", "formal").strip().lower()
EXACT_TOLERANCE = 1e-10

# Convert one array cell of the target distribution into a readable state assignment for auditing in the distribution_cells sheet.
def target_cell_assignment(index: tuple[int, ...]) -> dict[Node, str]:
    return {
        node: INDEX_TO_STATE[node][state_index]
        for node, state_index in zip(TARGET, index)
    }

# Store all state cells of the target joint distribution for one method, rather than retaining only the target event.
# The current TARGET=(INT, DISC, MVS) contains 3×2×3=18 state cells.
# Store all target-joint-distribution cells and their cell-wise errors.
def append_distribution_cells(
    rows: list[dict[str, object]],
    *,
    parameter_id: int,
    method: str,
    order_id: str,
    truth: np.ndarray,
    estimate: np.ndarray,
    protocol_sha256: str,
) -> None:
    for index in np.ndindex(truth.shape):
        assignment = target_cell_assignment(index)
        rows.append({
            "parameter_id": parameter_id,
            "method": method,
            "order_id": order_id,
            "target_assignment_json": json.dumps(assignment, sort_keys=True),
            "p_true": float(truth[index]),
            "p_estimate": float(estimate[index]),
            "absolute_error": float(abs(estimate[index] - truth[index])),
            "protocol_sha256": protocol_sha256,
        })

# Run the known-parameter experiment: each parameter set produces one J result, four FGCR results, and one IC result, for a total of six rows.
# Coordinate the exact comparison among the oracle, four-order FGCR, and IC under known parameters.
def run_experiment1(
    project_root: Path,
    run_mode: str,
    experiment_code_sha256: str,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, str]:
    project_root = Path(project_root)
    settings = get_settings(run_mode)

    # The parameter bank always generates/verifies the complete sets 0–99; quick mode only selects the first three sets specified by the protocol.
    bank = ensure_shared_parameter_bank(project_root / "data" / "alarm_parameter_sets.xlsx")
    selected_ids = tuple(int(value) for value in settings["parameter_ids"])

    # # Structure hashes cover the four H_R graphs and the G_R used by IC, ensuring that outputs cannot accidentally reuse outdated structures.
    structure_hashes = {
        "J_G": edge_sha256(ALARM_DAG),
        **{
            f"FGCR_{order_id}": edge_sha256(result.H_R)
            for order_id, result in OBSERVATIONAL_STRUCTURES.items()
        },
        "IC_G_R": edge_sha256(G_R),
    }
    experiment_payload = {
        "truth": "ancestral_enumeration_oracle_checked_by_VE",
        "fgcr_cpt_source": "exact_family_conditionals_from_source_BN",
        "ic_graph": "original_induced_subgraph_G_R",
        "target": TARGET,
        "target_event": TARGET_EVENT,
        "condition": CONDITION,
        "exact_tolerance": EXACT_TOLERANCE,
        "order_difference": "maximum pairwise TVD among four FGCR target distributions",
    }
    protocol_sha256 = experiment_protocol_sha256(
        1,
        run_mode,
        bank.content_sha256,
        structure_hashes,
        experiment_payload,
        experiment_code_sha256,
    )

    run_rows: list[dict[str, object]] = []
    cell_rows: list[dict[str, object]] = []
    order_rows: list[dict[str, object]] = []

    for parameter_id in selected_ids:
        # All methods use the original ALARM CPTs from the same parameter_id, eliminating interference caused by inconsistent parameter sets.
        source_cpts = bank.cpts(parameter_id)
        parameter_sha256 = cpts_sha256(source_cpts, PARENT_ORDER)

        # Independent ancestral enumeration is the primary ground truth; VE is used only to cross-check that the oracle implementation is correct.
        truth = ancestral_enumeration_query(
            source_cpts, ALARM_DAG, PARENT_ORDER, TARGET, CONDITION
        )
        truth_ve = ve_query(source_cpts, PARENT_ORDER, TARGET, CONDITION)
        oracle_errors = distribution_errors(truth, truth_ve)
        if max(oracle_errors.values()) > EXACT_TOLERANCE:
            raise AssertionError(f"Oracle/VE mismatch for parameter {parameter_id}")
        p_true = event_probability(truth, TARGET, TARGET_EVENT)

         # Query P(C) separately to identify potential numerical instability caused by extremely small evidence probabilities.
        condition_distribution = ve_query(
            source_cpts, PARENT_ORDER, CONDITION_NODES
        )
        p_condition = event_probability(
            condition_distribution, CONDITION_NODES, CONDITION
        )
        # J-Method is the independent oracle on the original graph. Store it explicitly as a zero-error row so that all six methods can be compared uniformly in the result table.
        run_rows.append({
            "experiment_id": 1,
            "run_mode": run_mode,
            "parameter_id": parameter_id,
            "method": "J",
            "order_id": "none",
            "graph_source": "original_G_independent_oracle",
            "edge_count": len(ALARM_DAG.edges),
            "parameter_dimension": parameter_dimension(PARENT_ORDER),
            "p_true": p_true,
            "p_estimate": p_true,
            "error": 0.0,
            "absolute_error": 0.0,
            "relative_bias": 0.0,
            "tvd": 0.0,
            "max_abs_error": 0.0,
            "condition_probability": p_condition,
            "parameter_sha256": parameter_sha256,
            "parameter_bank_sha256": bank.content_sha256,
            "structure_sha256": edge_sha256(ALARM_DAG),
            "protocol_sha256": protocol_sha256,
        })
        append_distribution_cells(
            cell_rows,
            parameter_id=parameter_id,
            method="J",
            order_id="none",
            truth=truth,
            estimate=truth,
            protocol_sha256=protocol_sha256,
        )

        # # IC uses the optimal family-wise conditional CPTs on G_R; therefore, any remaining error comes only from structural misspecification.
        ic_cpts, ic_parent_order = compile_conditional_cpts(
            source_cpts, PARENT_ORDER, G_R, CONDITION
        )
        ic_distribution = ve_query(ic_cpts, ic_parent_order, TARGET)
        ic_metrics = distribution_errors(truth, ic_distribution)
        p_ic = event_probability(ic_distribution, TARGET, TARGET_EVENT)
        ic_error = p_ic - p_true
        run_rows.append({
            "experiment_id": 1,
            "run_mode": run_mode,
            "parameter_id": parameter_id,
            "method": "IC",
            "order_id": "none",
            "graph_source": "G_R",
            "edge_count": len(G_R.edges),
            "parameter_dimension": parameter_dimension(G_R),
            "p_true": p_true,
            "p_estimate": p_ic,
            "error": ic_error,
            "absolute_error": abs(ic_error),
            "relative_bias": safe_relative_bias(ic_error, p_true),
            **ic_metrics,
            "condition_probability": p_condition,
            "parameter_sha256": parameter_sha256,
            "parameter_bank_sha256": bank.content_sha256,
            "structure_sha256": edge_sha256(G_R),
            "protocol_sha256": protocol_sha256,
        })
        append_distribution_cells(
            cell_rows,
            parameter_id=parameter_id,
            method="IC",
            order_id="none",
            truth=truth,
            estimate=ic_distribution,
            protocol_sha256=protocol_sha256,
        )

        fgcr_event_values: list[float] = []
        fgcr_distributions: list[np.ndarray] = []
        for order_id, result in OBSERVATIONAL_STRUCTURES.items():
            # Each H_R independently compiles its own local CPTs exactly from the same source BN and then performs an independent VE query.
            hr_cpts, hr_parent_order = compile_conditional_cpts(
                source_cpts, PARENT_ORDER, result.H_R, CONDITION
            )
            estimate = ve_query(hr_cpts, hr_parent_order, TARGET)
            metrics = distribution_errors(truth, estimate)
            p_estimate = event_probability(estimate, TARGET, TARGET_EVENT)
            error = p_estimate - p_true
            if max(metrics.values()) > EXACT_TOLERANCE or abs(error) > EXACT_TOLERANCE:
                raise AssertionError(
                    f"Exact FGCR failed: parameter={parameter_id}, order={order_id}"
                )
            fgcr_event_values.append(p_estimate)
            fgcr_distributions.append(estimate)
            run_rows.append({
                "experiment_id": 1,
                "run_mode": run_mode,
                "parameter_id": parameter_id,
                "method": "FGCR-C",
                "order_id": order_id,
                "graph_source": "FGCR_generated_H_R",
                "beta": ",".join(result.orders.beta),
                "alpha_C": ",".join(result.orders.alpha_C),
                "alpha_R": ",".join(result.orders.alpha_R),
                "edge_count": len(result.H_R.edges),
                "fill_edge_count": len(result.added_edges_R),
                "parameter_dimension": parameter_dimension(result.H_R),
                "p_true": p_true,
                "p_estimate": p_estimate,
                "error": error,
                "absolute_error": abs(error),
                "relative_bias": safe_relative_bias(error, p_true),
                **metrics,
                "condition_probability": p_condition,
                "parameter_sha256": parameter_sha256,
                "parameter_bank_sha256": bank.content_sha256,
                "structure_sha256": edge_sha256(result.H_R),
                "protocol_sha256": protocol_sha256,
            })
            append_distribution_cells(
                cell_rows,
                parameter_id=parameter_id,
                method="FGCR-C",
                order_id=order_id,
                truth=truth,
                estimate=estimate,
                protocol_sha256=protocol_sha256,
            )

         # Under the frozen protocol, OD is defined as the maximum pairwise TVD among the four complete target distributions.
        order_tvd = max(
            distribution_errors(first, second)["tvd"]
            for first, second in combinations(fgcr_distributions, 2)
        )
         # The max-minus-min difference of the scalar target event is retained only as an auxiliary audit quantity and does not replace the formal OD.
        event_difference = max(fgcr_event_values) - min(fgcr_event_values)
        if order_tvd > EXACT_TOLERANCE or event_difference > EXACT_TOLERANCE:
            raise AssertionError(f"FGCR order difference exceeded tolerance: {parameter_id}")
        order_rows.append({
            "parameter_id": parameter_id,
            "order_difference_tvd": order_tvd,
            "max_minus_min_event_probability": event_difference,
            "passed_le_1e_10": order_tvd <= EXACT_TOLERANCE,
        })

    runs = pd.DataFrame(run_rows)
    distribution_cells = pd.DataFrame(cell_rows)
    order_differences = pd.DataFrame(order_rows)
    summary = (
        runs.groupby(["method", "order_id"], sort=False)
        .agg(
            n_parameter_sets=("parameter_id", "nunique"),
            mean_absolute_error=("absolute_error", "mean"),
            max_absolute_error=("absolute_error", "max"),
            mean_tvd=("tvd", "mean"),
            max_tvd=("tvd", "max"),
            mean_max_cell_error=("max_abs_error", "mean"),
            max_max_cell_error=("max_abs_error", "max"),
            mean_relative_bias=("relative_bias", "mean"),
        )
        .reset_index()
    )
    protocol = experiment_protocol_frame(
        1,
        run_mode,
        bank.content_sha256,
        structure_hashes,
        experiment_payload,
        experiment_code_sha256,
        protocol_sha256,
    )
    return (
        runs,
        distribution_cells,
        order_differences,
        summary,
        protocol,
        protocol_sha256,
    )


## 3. Output and Plotting


In [3]:
# Write the numerical results, complete target distributions, structures, and protocol of Experiment 1 into a single workbook.
def save_experiment1(
    project_root: Path,
    runs: pd.DataFrame,
    distribution_cells: pd.DataFrame,
    order_differences: pd.DataFrame,
    summary: pd.DataFrame,
    protocol: pd.DataFrame,
    protocol_sha256: str,
) -> Path:
    output = Path(project_root) / "result" / "01_known_parameter_inference.xlsx"
    write_excel_atomic(output, {
        "runs": runs,
        "distribution_cells": distribution_cells,
        "order_difference": order_differences,
        "summary": summary,
        "structures": OBSERVATIONAL_STRUCTURE_AUDIT,
        "protocol": protocol,
        "environment": pd.DataFrame([environment_record()]),
    })
    verify_output_protocol(output, protocol_sha256)
    return output

# # Plot the logarithmic distribution of absolute errors for the four FGCR orders and IC.
def plot_experiment1(project_root: Path, runs: pd.DataFrame) -> Path:
    os.environ.setdefault("MPLCONFIGDIR", str(Path(project_root) / "result" / "matplotlib_cache"))
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    output = Path(project_root) / "result" / "01_known_parameter_absolute_error.png"
    #  J-Method provides p_true and is therefore excluded from the estimation-error plot.
    labels = [f"C-Method_{index}"for index in range(1, 5)] + ["IC"]

    values = [runs.loc[(runs["method"] == "FGCR-C")& (runs["order_id"] == f"order_{index}"), "absolute_error",].to_numpy(float)
        for index in range(1, 5)] + [ runs.loc[runs["method"] == "IC", "absolute_error", ].to_numpy(float) ]

    # A logarithmic axis cannot display zero values.
    # Use 1e-20 only in the plotting copy to represent zero errors on the log scale;
    # the original numerical results are not truncated or modified.
    plot_floor = 1e-20
    #plot_floor = np.finfo(float).tiny
    fig, ax = plt.subplots(figsize=(9.2, 5.0), constrained_layout=True)
    rng = np.random.default_rng(20260825)
    for x, (label, method_values) in enumerate(zip(labels, values), 1):
        shown = np.maximum(method_values, plot_floor)
        jitter = rng.uniform(-0.10, 0.10, len(shown))
        ax.scatter(np.full(len(shown), x) + jitter, shown, s=22, alpha=0.55)
        ax.hlines(np.median(shown), x - 0.18, x + 0.18, linewidth=2.2)
    ax.axhline(EXACT_TOLERANCE, linestyle="--", linewidth=1.1, label=r"$10^{-10}$ tolerance")
    ax.set_yscale("log")
    # Fix a reasonable display range so that zero errors do not expand the vertical axis down to approximately 1e-308.
    ax.set_ylim(plot_floor, 1.0)
    ax.set_xticks(range(1, len(labels) + 1), labels)
    ax.set_ylabel("Absolute error of target-event probability")
    ax.set_title("Known-parameter conditional inference")
    ax.legend(frameon=False)
    fig.savefig(output, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return output


## 4. Execution


In [4]:
EXPERIMENT1_CODE_SHA256 = tagged_code_sha256(
    PROJECT_ROOT / "01_Known_Parameter_Inference.ipynb", "experiment1_core"
)
(
    experiment1_runs,
    experiment1_distribution_cells,
    experiment1_order_differences,
    experiment1_summary,
    experiment1_protocol,
    experiment1_protocol_sha256,
) = run_experiment1(PROJECT_ROOT, RUN_MODE, EXPERIMENT1_CODE_SHA256)
experiment1_workbook = save_experiment1(
    PROJECT_ROOT,
    experiment1_runs,
    experiment1_distribution_cells,
    experiment1_order_differences,
    experiment1_summary,
    experiment1_protocol,
    experiment1_protocol_sha256,
)
experiment1_figure = plot_experiment1(PROJECT_ROOT, experiment1_runs)
print(experiment1_summary.to_string(index=False))
print(f"Workbook: {experiment1_workbook}")
print(f"Figure:   {experiment1_figure}")


method order_id  n_parameter_sets  mean_absolute_error  max_absolute_error     mean_tvd      max_tvd  mean_max_cell_error  max_max_cell_error  mean_relative_bias
     J     none               100         0.000000e+00        0.000000e+00 0.000000e+00 0.000000e+00         0.000000e+00        0.000000e+00        0.000000e+00
    IC     none               100         2.219408e-03        3.247246e-02 2.223380e-02 8.998281e-02         9.266335e-03        3.482318e-02        9.279354e-03
FGCR-C  order_1               100         3.469420e-17        3.608225e-16 4.115516e-16 1.120848e-15         2.574330e-16        9.992007e-16        4.567048e-17
FGCR-C  order_2               100         3.196079e-17        3.885781e-16 4.040837e-16 1.048315e-15         2.502859e-16        9.992007e-16        3.481680e-17
FGCR-C  order_3               100         3.263652e-17        3.330669e-16 3.995756e-16 1.082251e-15         2.533390e-16        9.436896e-16        2.716162e-17
FGCR-C  order_4             

In [8]:
# from pathlib import Path
# import pandas as pd

# PROJECT_ROOT = Path.cwd().resolve()
# workbook_path = PROJECT_ROOT / "result" / "01_known_parameter_inference.xlsx"

# # 必须使用与绘图函数内部期望一致的 sheet 名称，通常为 "runs"
# runs = pd.read_excel(workbook_path, sheet_name="runs")  

# # 调用绘图函数（确保 plot_experiment1 已定义）
# figure_path = plot_experiment1(PROJECT_ROOT, runs)
# print(f"Updated figure: {figure_path}")

Updated figure: D:\jupyternoteboook\FCGR算法实验\条件概率实验\条件概率实验（alarm)\result\01_known_parameter_absolute_error.png
